In [1]:
print(1)

1


In [2]:
from pathlib import Path

import pandas as pd
from scipy.stats import ttest_ind, ttest_rel, wilcoxon
from statsmodels.stats.multitest import multipletests

RESULTS = Path("..") / "results"
SUMMARY_TSV = RESULTS / "lpm_paper10_results_current_check_summary.tsv"
LONG_TSV = RESULTS / "lpm_paper10_results_current_check_long.tsv"

# Comparison: Scratch vs FT Morgan learned fixmol only.
VARIANT_NAME = "FT Morgan learned fixmol"
SCRATCH_FAMILY = "scratch_target_only"
VARIANT_FAMILY = "finetune_morgan_learned_fixed_updated_embeddings"
SCRATCH_COL = "scratch_target_only_test_rmse_mean"
VARIANT_COL = "finetune_morgan_learned_fixmol_test_rmse_mean"

# Display names matching the 11-dataset paper table (exclude VCPI-0002: no test split).
DISPLAY_NAME = {
    "lincs_phase1": "LINCS Phase I",
    "lincs_phase2": "LINCS Phase II",
    "novartis": "Novartis DRUG-seq",
    "vcpi_0001": "VCPI vcpi-0001",
    "op3": "OP3",
    "tahoe100": "Tahoe-100M",
    "cigs_tcm": "CIGS TCM",
    "dilimap_train": "DILImap train",
    "gdpx2": "Ginkgo GDPx2",
    "sciplex": "sci-Plex",
    "cigs_mce": "CIGS MCE",
}
TABLE11_ORDER = list(DISPLAY_NAME)

In [3]:
summary = pd.read_csv(SUMMARY_TSV, sep="\t")
table11 = (
    summary.set_index("dataset_slug")
    .loc[TABLE11_ORDER]
    .assign(dataset_display=lambda d: d.index.map(DISPLAY_NAME))
    .reset_index()
)

# Δ = mean_test_RMSE(Scratch) − mean_test_RMSE(variant); positive ⇒ variant better.
deltas = pd.DataFrame(
    {
        "dataset": table11["dataset_display"],
        "scratch_rmse": table11[SCRATCH_COL],
        "variant_rmse": table11[VARIANT_COL],
    }
)
deltas["delta"] = deltas["scratch_rmse"] - deltas["variant_rmse"]
deltas["variant_better"] = deltas["delta"] > 0

display(deltas.round(6))

# One-sided: H1 that variant beats Scratch (Δ > 0) across datasets.
stat, p_greater = wilcoxon(deltas["delta"], alternative="greater", zero_method="wilcox")
_, p_two_sided = wilcoxon(deltas["delta"], alternative="two-sided", zero_method="wilcox")

print(f"\nAggregate Wilcoxon signed-rank: Scratch vs {VARIANT_NAME}")
print(f"  n datasets          = {len(deltas)}")
print(f"  n where variant wins = {int(deltas['variant_better'].sum())}")
print(f"  mean Δ (Scratch−var) = {deltas['delta'].mean():.6f}")
print(f"  median Δ             = {deltas['delta'].median():.6f}")
print(f"  W statistic          = {stat:.4f}")
print(f"  p (greater, one-sided) = {p_greater:.4g}")
print(f"  p (two-sided)          = {p_two_sided:.4g}")

,dataset,scratch_rmse,variant_rmse,delta,variant_better
0,LINCS Phase I,0.63260,0.62145,0.01115,True
1,LINCS Phase II,0.48857,0.48137,0.00720,True
2,Novartis DRUG-seq,0.54082,0.53713,0.00369,True
3,VCPI vcpi-0001,0.57072,0.57065,0.00007,True
4,OP3,0.34781,0.34305,0.00476,True
5,Tahoe-100M,0.73803,0.73537,0.00266,True
6,CIGS TCM,0.40805,0.40767,0.00038,True
7,DILImap train,0.60953,0.60946,0.00007,True
8,Ginkgo GDPx2,0.43307,0.43800,-0.00493,False
9,sci-Plex,0.68454,0.68991,-0.00537,False



Aggregate Wilcoxon signed-rank: Scratch vs FT Morgan learned fixmol
  n datasets          = 11
  n where variant wins = 8
  mean Δ (Scratch−var) = 0.001458
  median Δ             = 0.000380
  W statistic          = 44.0000
  p (greater, one-sided) = 0.1768
  p (two-sided)          = 0.3535


In [4]:
# Per-dataset Welch t-tests on 10 seed test RMSEs, Holm-corrected across datasets.
# H1: mean Scratch RMSE > mean variant RMSE (variant better). Unpaired (equal_var=False).

long = pd.read_csv(LONG_TSV, sep="\t")

rows = []
for slug in TABLE11_ORDER:
    scratch = long.loc[
        (long["model_family"] == SCRATCH_FAMILY) & (long["dataset_slug"] == slug),
        "test_rmse",
    ].astype(float)
    variant = long.loc[
        (long["model_family"] == VARIANT_FAMILY) & (long["dataset_slug"] == slug),
        "test_rmse",
    ].astype(float)
    assert len(scratch) == 10 and len(variant) == 10, (slug, len(scratch), len(variant))

    res = ttest_ind(scratch, variant, equal_var=False, alternative="greater")
    rows.append(
        {
            "dataset": DISPLAY_NAME[slug],
            "scratch_mean": scratch.mean(),
            "variant_mean": variant.mean(),
            "delta": scratch.mean() - variant.mean(),
            "scratch_std": scratch.std(ddof=1),
            "variant_std": variant.std(ddof=1),
            "t": res.statistic,
            "df": res.df,
            "p_raw": res.pvalue,
        }
    )

welch = pd.DataFrame(rows)
reject, p_holm, _, _ = multipletests(welch["p_raw"], alpha=0.05, method="holm")
welch["p_holm"] = p_holm
welch["sig_holm_0.05"] = reject

display(
    welch.style.format(
        {
            "scratch_mean": "{:.6f}",
            "variant_mean": "{:.6f}",
            "delta": "{:.6f}",
            "scratch_std": "{:.6f}",
            "variant_std": "{:.6f}",
            "t": "{:.3f}",
            "df": "{:.2f}",
            "p_raw": "{:.4g}",
            "p_holm": "{:.4g}",
        }
    )
)

print(f"\nWelch + Holm: Scratch vs {VARIANT_NAME}")
print(f"  datasets with p_holm < 0.05: {int(welch['sig_holm_0.05'].sum())} / {len(welch)}")
print(
    "  significant:",
    ", ".join(welch.loc[welch["sig_holm_0.05"], "dataset"]) or "(none)",
)

,dataset,scratch_mean,variant_mean,delta,scratch_std,variant_std,t,df,p_raw,p_holm,sig_holm_0.05
0,LINCS Phase I,0.632600,0.621450,0.011150,0.001382,0.001134,19.729,17.34,1.275e-13,1.275e-12,True
1,LINCS Phase II,0.488570,0.481370,0.007200,0.000374,0.000440,39.423,17.55,6.985e-19,7.683e-18,True
2,Novartis DRUG-seq,0.540820,0.537130,0.003690,0.000563,0.000640,13.692,17.72,3.67e-11,3.303e-10,True
3,VCPI vcpi-0001,0.570720,0.570650,0.000070,0.000063,0.000071,2.333,17.78,0.01579,0.09475,False
4,OP3,0.347810,0.343050,0.004760,0.001061,0.001338,8.816,17.11,4.499e-08,3.599e-07,True
5,Tahoe-100M,0.738030,0.735370,0.002660,0.000392,0.001312,6.145,10.59,4.264e-05,0.0002985,True
6,CIGS TCM,0.408050,0.407670,0.000380,0.000255,0.000591,1.868,12.24,0.04297,0.2149,False
7,DILImap train,0.609530,0.609460,0.000070,0.001364,0.000804,0.140,14.59,0.4454,1,False
8,Ginkgo GDPx2,0.433070,0.438000,-0.004930,0.003063,0.002340,-4.045,16.83,0.9996,1,False
9,sci-Plex,0.684540,0.689910,-0.005370,0.000566,0.002802,-5.940,9.73,0.9999,1,False



Welch + Holm: Scratch vs FT Morgan learned fixmol
  datasets with p_holm < 0.05: 5 / 11
  significant: LINCS Phase I, LINCS Phase II, Novartis DRUG-seq, OP3, Tahoe-100M


In [6]:
# Per-dataset Welch t-tests on 10 seed test RMSEs, Holm-corrected across datasets.
# H1: mean Scratch RMSE > mean variant RMSE (variant better). Unpaired (equal_var=False).

long = pd.read_csv(LONG_TSV, sep="\t")

rows = []
for slug in TABLE11_ORDER:
    scratch = long.loc[
        (long["model_family"] == SCRATCH_FAMILY) & (long["dataset_slug"] == slug),
        "test_rmse",
    ].astype(float)
    variant = long.loc[
        (long["model_family"] == VARIANT_FAMILY) & (long["dataset_slug"] == slug),
        "test_rmse",
    ].astype(float)
    assert len(scratch) == 10 and len(variant) == 10, (slug, len(scratch), len(variant))

    res = ttest_ind(scratch, variant, equal_var=False, alternative="two-sided")
    rows.append(
        {
            "dataset": DISPLAY_NAME[slug],
            "scratch_mean": scratch.mean(),
            "variant_mean": variant.mean(),
            "delta": scratch.mean() - variant.mean(),
            "scratch_std": scratch.std(ddof=1),
            "variant_std": variant.std(ddof=1),
            "t": res.statistic,
            "df": res.df,
            "p_raw": res.pvalue,
        }
    )

welch = pd.DataFrame(rows)
reject, p_holm, _, _ = multipletests(welch["p_raw"], alpha=0.05, method="holm")
welch["p_holm"] = p_holm
welch["sig_holm_0.05"] = reject

display(
    welch.style.format(
        {
            "scratch_mean": "{:.6f}",
            "variant_mean": "{:.6f}",
            "delta": "{:.6f}",
            "scratch_std": "{:.6f}",
            "variant_std": "{:.6f}",
            "t": "{:.3f}",
            "df": "{:.2f}",
            "p_raw": "{:.4g}",
            "p_holm": "{:.4g}",
        }
    )
)

print(f"\nWelch + Holm: Scratch vs {VARIANT_NAME}")
print(f"  datasets with p_holm < 0.05: {int(welch['sig_holm_0.05'].sum())} / {len(welch)}")
print(
    "  significant:",
    ", ".join(welch.loc[welch["sig_holm_0.05"], "dataset"]) or "(none)",
)

,dataset,scratch_mean,variant_mean,delta,scratch_std,variant_std,t,df,p_raw,p_holm,sig_holm_0.05
0,LINCS Phase I,0.632600,0.621450,0.011150,0.001382,0.001134,19.729,17.34,2.55e-13,2.55e-12,True
1,LINCS Phase II,0.488570,0.481370,0.007200,0.000374,0.000440,39.423,17.55,1.397e-18,1.537e-17,True
2,Novartis DRUG-seq,0.540820,0.537130,0.003690,0.000563,0.000640,13.692,17.72,7.34e-11,6.606e-10,True
3,VCPI vcpi-0001,0.570720,0.570650,0.000070,0.000063,0.000071,2.333,17.78,0.03158,0.09475,False
4,OP3,0.347810,0.343050,0.004760,0.001061,0.001338,8.816,17.11,8.998e-08,7.198e-07,True
5,Tahoe-100M,0.738030,0.735370,0.002660,0.000392,0.001312,6.145,10.59,8.527e-05,0.0005969,True
6,CIGS TCM,0.408050,0.407670,0.000380,0.000255,0.000591,1.868,12.24,0.08594,0.1719,False
7,DILImap train,0.609530,0.609460,0.000070,0.001364,0.000804,0.140,14.59,0.8907,0.8907,False
8,Ginkgo GDPx2,0.433070,0.438000,-0.004930,0.003063,0.002340,-4.045,16.83,0.0008564,0.003425,True
9,sci-Plex,0.684540,0.689910,-0.005370,0.000566,0.002802,-5.940,9.73,0.0001595,0.0009572,True



Welch + Holm: Scratch vs FT Morgan learned fixmol
  datasets with p_holm < 0.05: 8 / 11
  significant: LINCS Phase I, LINCS Phase II, Novartis DRUG-seq, OP3, Tahoe-100M, Ginkgo GDPx2, sci-Plex, CIGS MCE
